# FIFA 22 Player Analysis: Performance, Ranking & Preferred Foot Classification

**Author:** Isabel Garcia  
**Tools:** Python, pandas, scikit-learn, statsmodels, seaborn, matplotlib

## Overview
This project analyzes a dataset of 19,630 professional soccer players (men's and women's) from FIFA 22.
The analysis is split into two parts:

1. **Regression Analysis** — Using player attributes (passing, attacking, defending, skill) to predict player rank via OLS and scikit-learn Linear Regression
2. **KNN Classification** — Predicting a player's preferred foot (left/right) using skill-based features and K-Nearest Neighbors

Key questions explored:
- Which player attributes are the strongest predictors of rank?
- Can we reliably classify a player's preferred foot from performance stats?
- How does class imbalance affect model performance?

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import (
    mean_squared_error, r2_score,
    accuracy_score, confusion_matrix,
    classification_report
)
import sklearn.metrics as metrics
import statsmodels.formula.api as smf
import statsmodels.api as sm

sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 2. Load & Explore the Data

In [ ]:
fifa = pd.read_csv("shared/data/fifa22.csv")
print(f"Dataset shape: {fifa.shape}")
fifa.head()

**Unit of analysis:** Each row represents one individual professional soccer player. Features include rank, gender, wage, position, nationality, club, league, preferred foot, and aggregated skill ratings (shooting, passing, dribbling, etc.).

In [ ]:
# Dataset info
fifa.info()

In [ ]:
# Gender breakdown
gender_counts = fifa['gender'].value_counts()
print(f"Male players:   {gender_counts['M']:,}")
print(f"Female players: {gender_counts['F']:,}")
print(f"\nDataset is heavily male-dominated ({gender_counts['M']/len(fifa)*100:.1f}% male).")

In [ ]:
# Missing values
missing = fifa.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0])

**Missing value patterns:**
- `position`, `club`, and `league` each have 452 missing values — likely players currently without a contract
- `shooting`, `passing`, and `dribbling` share the same missingness — likely goalkeepers or unrated players
- `wage_eur` and `log_wage` are missing for all female players (wages not tracked in FIFA 22 for women's teams)

We drop rows missing `passing` since it is a key predictor.

In [ ]:
fifa_clean = fifa.dropna(subset=['passing'])
print(f"Cleaned dataset shape: {fifa_clean.shape}")
print(f"Dropped {len(fifa) - len(fifa_clean):,} rows with missing 'passing' values.")

## 3. Part 1 — Regression: Predicting Player Rank

We use four player attributes — `passing`, `attacking`, `defending`, and `skill` — to predict a player's overall `rank`.

In [ ]:
# OLS Regression via statsmodels
rank_model = smf.ols('rank ~ passing + attacking + defending + skill', data=fifa_clean).fit()
rank_model.summary()

**Key findings from OLS:**
- **R² = 0.705** — the model explains 70.5% of the variance in player rank
- `attacking`, `passing`, and `defending` are statistically significant at the 5% level (p < 0.05)
- `skill` has a p-value of 0.465 — not statistically significant; its confidence interval includes zero, meaning we cannot determine its true directional effect on rank

In [ ]:
# Residuals vs Fitted Values
fitted_values = rank_model.fittedvalues
residuals = rank_model.resid

plt.figure(figsize=(10, 6))
plt.scatter(fitted_values, residuals, alpha=0.4, color='steelblue')
plt.axhline(y=0, color='black', linestyle='--')
plt.xlabel('Fitted Values')
plt.ylabel('Residuals')
plt.title('Residuals vs Fitted Values — Rank Regression Model')
plt.grid(True)
plt.show()

**Residual plot observations:** Values are somewhat clustered rather than randomly scattered, suggesting possible non-linearity. There are also notable outliers around fitted values of 65–70, which may represent elite players whose rank is harder to predict from skill attributes alone.

### 3.1 scikit-learn Linear Regression (Train/Test Split)

In [ ]:
X = fifa_clean[['passing', 'attacking', 'defending', 'skill']]
Y = fifa_clean[['rank']]

X_train, X_valid, Y_train, Y_valid = train_test_split(X, Y, test_size=0.25, random_state=123)
print(f"Training set size:   {X_train.shape[0]:,}")
print(f"Validation set size: {X_valid.shape[0]:,}")

In [ ]:
# Fit sklearn model
model_sk = LinearRegression()
model_sk.fit(X_train, Y_train)

coef_df = pd.DataFrame({
    'Feature': X.columns,
    'sklearn Coefficient': model_sk.coef_[0].round(5)
})
print(f"Intercept: {model_sk.intercept_[0]:.4f}\n")
print(coef_df.to_string(index=False))

The sklearn and statsmodels coefficients are nearly identical (differences < 0.002), confirming both implementations converge to the same estimates. `attacking` has the largest positive effect on rank (~0.61 per unit increase).

In [ ]:
# Predictions and model evaluation
y_pred = model_sk.predict(X_valid)

rmse = np.sqrt(mean_squared_error(Y_valid, y_pred))
r2 = r2_score(Y_valid, y_pred)
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")

In [ ]:
# Actual vs Predicted plot
plt.figure(figsize=(8, 6))
plt.scatter(Y_valid, y_pred, alpha=0.4, color='steelblue')
plt.plot([Y_valid.min(), Y_valid.max()], [Y_valid.min(), Y_valid.max()], 'r--', label='Perfect Prediction')
plt.xlabel('Actual Rank')
plt.ylabel('Predicted Rank')
plt.title('Actual vs Predicted Player Rank')
plt.legend()
plt.grid(True)
plt.show()

**Model summary:** With an RMSE of ~3.74, the model predicts a player's rank within about 3–4 positions on average across 17,450 players. Given the scale of the dataset and the simplicity of the model (4 features), this is a solid result.

## 4. Part 2 — KNN Classification: Predicting Preferred Foot

Can we predict whether a player is left- or right-footed based on their skill attributes?

In [ ]:
# Class distribution
foot_counts = fifa_clean['preferred_foot'].value_counts()
print(foot_counts)
print(f"\nRight-footed: {foot_counts['Right']/foot_counts.sum()*100:.1f}%")
print(f"Left-footed:  {foot_counts['Left']/foot_counts.sum()*100:.1f}%")
print("\nNote: A naive classifier that always predicts 'Right' would be correct ~74.8% of the time.")

In [ ]:
# Visualize class imbalance
plt.figure(figsize=(6, 4))
sns.barplot(x=foot_counts.index, y=foot_counts.values, palette=['steelblue', 'salmon'])
plt.title('Preferred Foot Distribution')
plt.xlabel('Preferred Foot')
plt.ylabel('Number of Players')
plt.show()

In [ ]:
# Features and target
features = ["shooting", "passing", "dribbling", "attacking",
            "defending", "skill", "movement", "power", "mentality", "goalkeeping"]

X = fifa_clean[features]
Y = fifa_clean['preferred_foot']

# Scale features (important for KNN)
sc = StandardScaler()
X_scaled = pd.DataFrame(sc.fit_transform(X), columns=X.columns)

X_train, X_test, Y_train, Y_test = train_test_split(X_scaled, Y, test_size=0.3, random_state=456)
print(f"Training: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")

In [ ]:
# Tune k: plot error rate for k=1 to 30
errors = []
for k in range(1, 31):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, Y_train)
    error = 1 - knn.score(X_test, Y_test)
    errors.append(error)

plt.figure(figsize=(12, 5))
plt.plot(range(1, 31), errors, marker='o', color='steelblue')
plt.title('KNN Classification Error vs. Number of Neighbors')
plt.xlabel('Number of Neighbors (k)')
plt.ylabel('Error Rate')
plt.grid(True)
plt.show()

best_k = np.argmin(errors) + 1
print(f"Lowest error at k={best_k}: {min(errors):.4f}")

In [ ]:
# Final model with k=10
knn = KNeighborsClassifier(n_neighbors=10)
knn.fit(X_train, Y_train)
Y_pred = knn.predict(X_test)

In [ ]:
# Confusion matrix
cm = confusion_matrix(Y_test, Y_pred, labels=['Left', 'Right'])

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Left', 'Predicted Right'],
            yticklabels=['Actual Left', 'Actual Right'])
plt.title('Confusion Matrix — Preferred Foot Classification')
plt.show()

In [ ]:
# Classification report
print(classification_report(Y_test, Y_pred))

## 5. Conclusions

### Part 1 — Rank Regression
- A linear model using `passing`, `attacking`, `defending`, and `skill` explains **70.5% of variance** in player rank (R² = 0.705)
- RMSE of ~3.74 means predictions are off by ~3–4 ranking positions on average
- `attacking` is the strongest predictor; `skill` is not statistically significant
- Both statsmodels and sklearn produce nearly identical coefficients, validating the estimates

### Part 2 — Preferred Foot Classification
- The KNN classifier (k=10) achieves **72% overall accuracy** — marginally better than the 74.8% naive baseline, but only because it correctly predicts right-footed players
- The model is severely biased: it correctly identifies left-footed players only **17% of the time** (recall = 0.17)
- The core issue is **class imbalance** — 74.8% of players are right-footed, so the model defaults toward the majority class
- Potential improvements: SMOTE oversampling, class weighting, or ensemble methods to address imbalance